# Movie Plot Map — embeddings + 2D visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thasup/chatgpt-api-notebook/blob/main/Movies_Plotting_with_Atlas.ipynb)

Embed the plots of 5,000 recent American movies, project them to 2D with UMAP, and plot them so movies with similar stories land near each other.

| Step | Tool |
|---|---|
| Embeddings | [OpenRouter](https://openrouter.ai) (OpenAI-compatible API) → `openai/text-embedding-3-small` |
| 2D projection | UMAP (cosine distance) |
| Charts | Plotly (interactive, also exported to `movie_map.html`) + Matplotlib (static PNG, renders on GitHub) |
| Optional | Nomic Atlas hosted map (needs `NOMIC_API_KEY`) |

**You need:** an `OPEN_ROUTER_API_KEY`.
- **Colab:** add it in the 🔑 *Secrets* panel and enable notebook access.
- **Local:** put `OPEN_ROUTER_API_KEY=...` in a `.env` file next to this notebook (already git-ignored).

**Cost:** about 3.7M tokens; the estimate cell below prints the exact number before anything is sent.

## 1. Install

In [ ]:
%pip install -q openai python-dotenv pandas numpy tenacity tiktoken umap-learn plotly matplotlib

## 2. Config

In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import tiktoken
from dotenv import dotenv_values
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_random_exponential

# Check available embedding models at https://openrouter.ai/models?output_modalities=embeddings
EMBEDDING_MODEL = "openai/text-embedding-3-small"
PRICE_PER_1M_TOKENS = 0.02  # USD, for the model above
MAX_TOKENS_PER_INPUT = 8000  # model limit is 8191
BATCH_SIZE = 100  # inputs per API request
N_MOVIES = 5000

DATASET_PATH = Path("movie_plot.csv")
DATASET_URL = "https://raw.githubusercontent.com/thasup/chatgpt-api-notebook/main/movie_plot.csv"
EMBEDDING_CACHE_PATH = Path("movie_embedding_cache.pkl")

In [ ]:
def load_api_key(name="OPEN_ROUTER_API_KEY"):
    """Read the key from Colab Secrets, then the environment, then a local .env file."""
    try:
        from google.colab import userdata  # only exists on Colab
        key = userdata.get(name)
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get(name) or dotenv_values(".env").get(name)
    if not key:
        raise RuntimeError(f"{name} not found. Add it to Colab Secrets or a .env file.")
    return key


client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_api_key())

## 3. Data

In [ ]:
if not DATASET_PATH.exists():
    print(f"Downloading dataset from {DATASET_URL} ...")
    df = pd.read_csv(DATASET_URL)
    df.to_csv(DATASET_PATH, index=False)
else:
    df = pd.read_csv(DATASET_PATH)

df.shape

In [ ]:
movies = (
    df[df["Origin/Ethnicity"] == "American"]
    .sort_values("Release Year", ascending=False)
    .head(N_MOVIES)
    .reset_index(drop=True)
)

# Collapse ~900 raw genre strings into the 10 most common primary genres + "other"
primary = movies["Genre"].fillna("unknown").str.split(r"[,/]").str[0].str.strip().str.lower()
top_genres = primary.value_counts().head(10).index
movies["genre_group"] = primary.where(primary.isin(top_genres), "other")

movies[["Release Year", "Title", "Director", "Genre", "genre_group"]].head()

## 4. Token count & cost estimate

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")  # tokenizer used by text-embedding-3-*


def prepare(text):
    """Flatten newlines and truncate to the model's input limit."""
    tokens = enc.encode(text.replace("\n", " "))
    return enc.decode(tokens[:MAX_TOKENS_PER_INPUT])


movie_plots = [prepare(p) for p in movies["Plot"]]
total_tokens = sum(len(enc.encode(p)) for p in movie_plots)
print(f"{len(movie_plots):,} plots, {total_tokens:,} tokens")
print(f"Estimated cost: ${total_tokens / 1_000_000 * PRICE_PER_1M_TOKENS:.2f}")

## 5. Embeddings (batched + cached)

Plots are sent in batches of `BATCH_SIZE` and cached on disk, so re-running only pays for plots not embedded yet.
On Colab the cache is lost when the runtime resets — download `movie_embedding_cache.pkl` or copy it to Drive if you want to keep it.

In [ ]:
try:
    with open(EMBEDDING_CACHE_PATH, "rb") as f:
        embedding_cache = pickle.load(f)
except FileNotFoundError:
    embedding_cache = {}


@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def embed_batch(texts, model=EMBEDDING_MODEL):
    response = client.embeddings.create(model=model, input=texts)
    if not response.data or len(response.data) != len(texts):
        raise RuntimeError(f"Expected {len(texts)} embeddings, got {len(response.data or [])}")
    return [d.embedding for d in sorted(response.data, key=lambda d: d.index)]


def embed_all(texts, model=EMBEDDING_MODEL):
    unique = list(dict.fromkeys(texts))  # a few plots are exact duplicates
    missing = [t for t in unique if (t, model) not in embedding_cache]
    print(f"{len(unique):,} unique plots: {len(unique) - len(missing):,} cached, {len(missing):,} to embed")
    for start in range(0, len(missing), BATCH_SIZE):
        batch = missing[start:start + BATCH_SIZE]
        for text, vector in zip(batch, embed_batch(batch, model)):
            embedding_cache[(text, model)] = vector
        with open(EMBEDDING_CACHE_PATH, "wb") as f:
            pickle.dump(embedding_cache, f)
        print(f"  embedded {min(start + BATCH_SIZE, len(missing)):,}/{len(missing):,}")
    return np.array([embedding_cache[(t, model)] for t in texts])

In [ ]:
# Smoke test: one call before spending on the full run
len(embed_batch(["Fried Chicken is the best!!"])[0])

In [ ]:
plot_embeddings = embed_all(movie_plots)
plot_embeddings.shape

## 6. Project to 2D with UMAP

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, metric="cosine", n_neighbors=15, min_dist=0.1, random_state=42)
coords = reducer.fit_transform(plot_embeddings)
movies["x"], movies["y"] = coords[:, 0], coords[:, 1]

## 7. Interactive map (Plotly)

Hover a point to see the movie. Click legend entries to toggle genres. Also saved as `movie_map.html`, a standalone page you can open in any browser or host on GitHub Pages.

In [ ]:
import plotly.express as px
import plotly.io as pio

# Colab needs its own renderer; elsewhere emit both the Plotly mimetype (JupyterLab/VS Code)
# and CDN-backed HTML (classic Jupyter, nbviewer) so saved outputs render in more places.
try:
    import google.colab  # noqa: F401
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype+notebook_connected"

genre_order = list(top_genres) + ["other"]
fig = px.scatter(
    movies,
    x="x",
    y="y",
    color="genre_group",
    category_orders={"genre_group": genre_order},
    hover_name="Title",
    hover_data={"Release Year": True, "Director": True, "Genre": True, "x": False, "y": False, "genre_group": False},
    opacity=0.7,
    title=f"{len(movies):,} American movie plots ({movies['Release Year'].min()}–{movies['Release Year'].max()}), embedded with {EMBEDDING_MODEL}",
    width=1000,
    height=750,
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(legend_title_text="Primary genre", xaxis_visible=False, yaxis_visible=False)
fig.write_html("movie_map.html", include_plotlyjs="cdn")
fig.show()

## 8. Static map (Matplotlib)

GitHub's notebook preview does not run JavaScript, so this PNG version is what shows up there.

In [ ]:
import matplotlib.pyplot as plt

fig_static, ax = plt.subplots(figsize=(12, 9))
cmap = plt.get_cmap("tab10")
other = movies[movies["genre_group"] == "other"]
ax.scatter(other["x"], other["y"], s=4, alpha=0.3, color="#cccccc", label=f"other ({len(other)})")  # background layer
for i, genre in enumerate(top_genres):
    subset = movies[movies["genre_group"] == genre]
    ax.scatter(subset["x"], subset["y"], s=6, alpha=0.7, color=cmap(i), label=f"{genre} ({len(subset)})")
ax.set_title("Movie plots in embedding space (UMAP, cosine)")
ax.set_xticks([])
ax.set_yticks([])
ax.legend(markerscale=3, fontsize=9, loc="best")
plt.tight_layout()
fig_static.savefig("movie_map.png", dpi=150)
plt.show()

## 9. Optional: Nomic Atlas hosted map

Uploads the embeddings + metadata to [atlas.nomic.ai](https://atlas.nomic.ai) and returns a link to a hosted, zoomable map with automatic topic labels.
Skipped unless `NOMIC_API_KEY` is set (Colab Secrets or `.env`). Check whether the map is public or private before sharing the link.

In [ ]:
try:
    nomic_key = load_api_key("NOMIC_API_KEY")
except RuntimeError:
    nomic_key = None
    print("NOMIC_API_KEY not set — skipping Nomic Atlas.")

if nomic_key:
    %pip install -q nomic
    import nomic
    from nomic import atlas

    nomic.login(nomic_key)
    atlas_data = movies[["Title", "Release Year", "Director", "Genre", "genre_group"]].copy()
    atlas_data["id"] = atlas_data.index.astype(str)
    dataset = atlas.map_data(
        embeddings=plot_embeddings,
        data=atlas_data,
        id_field="id",
        identifier="movie-plots",
    )
    print(dataset)